# 🚗 VehiclEye: Entrenamiento en Google Colab

**Duración total:** ~2 horas (con GPU gratis T4)

### Qué hace este notebook:
1. Verifica GPU y monta Drive
2. Instala dependencias
3. Clona el repositorio
4. Descarga imágenes REALES de vehículos (DuckDuckGo + validación)
5. Entrena EfficientNet-B0
6. Evalúa el modelo
7. Exporta a ONNX
8. Descarga el modelo final

---
> ⚡ **Ejecuta cada celda con Shift+Enter y espera a que termine antes de pasar a la siguiente.**

## 🔧 CELDA 1 — Verificar GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU: {gpu_name}  ({gpu_mem:.1f} GB)')
else:
    print('⚠️  No hay GPU. Ve a: Entorno de ejecución → Cambiar tipo → GPU (T4)')
    print('   Sin GPU el entrenamiento tardará 5-6 horas en vez de 90 min.')

print(f'✅ PyTorch {torch.__version__}')
print('\n✅ Todo listo para continuar.')

## 📦 CELDA 2 — Instalar dependencias

In [ ]:
# Dependencias de ML (PyTorch ya está en Colab)
!pip install -q timm albumentations

# Descarga de imágenes (DuckDuckGo, sin bloqueos)
!pip install -q duckduckgo-search requests pillow

# ONNX para exportar el modelo
!pip install -q onnx onnxruntime

import torch, timm, PIL, onnx
print(f'✅ torch={torch.__version__}  timm={timm.__version__}  PIL={PIL.__version__}')
print('✅ Todas las dependencias instaladas.')

## 📂 CELDA 3 — Clonar repositorio

In [ ]:
import os, subprocess

REPO_URL    = 'https://github.com/nick2331/Electiva_3.git'
REPO_BRANCH = 'claude/analyze-project-tech-oKyKr'
REPO_DIR    = '/content/Electiva_3'

if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
    print('✅ Repositorio clonado.')
else:
    !git -C {REPO_DIR} pull origin {REPO_BRANCH}
    print('✅ Repositorio actualizado.')

os.chdir(REPO_DIR)
print(f'📁 Directorio de trabajo: {os.getcwd()}')

## 🖼️ CELDA 4 — Descargar imágenes de vehículos

**Usa DuckDuckGo Images** (sin API key, sin bloqueos).
Incluye **validación automática** para descartar:
- Imágenes negras / siluetas de anime
- Imágenes corruptas
- Imágenes demasiado pequeñas
- Logos o iconos sin color

⏱️ Duración estimada: **20-30 minutos**

In [ ]:
import time, hashlib, requests
from pathlib import Path
from io import BytesIO
from PIL import Image
from duckduckgo_search import DDGS

# ─── Configuración ───────────────────────────────────────────────
IMAGES_PER_CLASS = 60          # Imágenes a descargar por clase
MIN_VALID        = 30          # Mínimo aceptable para entrenar
OUTPUT_DIR       = Path('ml/data/vehicleye_dataset')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Catálogo (20 clases) ────────────────────────────────────────
VEHICLE_CLASSES = [
    ('Toyota',     'Corolla'),
    ('Toyota',     'Hilux'),
    ('Chevrolet',  'Spark'),
    ('Chevrolet',  'Aveo'),
    ('Renault',    'Logan'),
    ('Renault',    'Sandero'),
    ('Renault',    'Stepway'),
    ('Mazda',      '3'),
    ('Mazda',      'CX-5'),
    ('Hyundai',    'Tucson'),
    ('Hyundai',    'Accent'),
    ('Kia',        'Picanto'),
    ('Kia',        'Rio'),
    ('Nissan',     'Frontier'),
    ('Nissan',     'Versa'),
    ('Ford',       'Fiesta'),
    ('Ford',       'Escape'),
    ('Volkswagen', 'Gol'),
    ('Volkswagen', 'Jetta'),
    ('Suzuki',     'Swift'),
]

# ─── Búsquedas variadas para cada modelo ────────────────────────
def get_queries(brand: str, model: str) -> list[str]:
    return [
        f'{brand} {model} automobile exterior photo',
        f'{brand} {model} car side view',
        f'{brand} {model} car front view',
        f'{brand} {model} car parked street',
    ]

# ─── Nombre de carpeta: toyota_corolla, mazda_cx5, etc. ─────────
def to_dir_name(brand: str, model: str) -> str:
    name = f'{brand}_{model}'.lower()
    return name.replace('-', '').replace(' ', '_').replace('.', '')

# ─── Validación: rechaza siluetas, anime, iconos ────────────────
def is_valid_car_image(data: bytes, min_px: int = 120) -> bool:
    try:
        img = Image.open(BytesIO(data)).convert('RGB')
        w, h = img.size
        if w < min_px or h < min_px:
            return False            # Demasiado pequeña
        pixels = list(img.getdata())
        total  = len(pixels)
        # Rechaza imágenes casi negras (siluetas de anime)
        black  = sum(1 for r, g, b in pixels if r < 40  and g < 40  and b < 40)
        # Rechaza imágenes casi blancas (logos, iconos vacíos)
        white  = sum(1 for r, g, b in pixels if r > 220 and g > 220 and b > 220)
        # Rechaza imágenes en escala de grises (sin color = no foto real)
        gray   = sum(1 for r, g, b in pixels if abs(r-g) < 10 and abs(g-b) < 10)
        if black / total > 0.55:   return False
        if white / total > 0.80:   return False
        if gray  / total > 0.90:   return False
        return True
    except Exception:
        return False

# ─── Descarga una clase completa ────────────────────────────────
def download_class(brand: str, model: str) -> int:
    dir_name  = to_dir_name(brand, model)
    class_dir = OUTPUT_DIR / dir_name
    class_dir.mkdir(parents=True, exist_ok=True)

    existing = len(list(class_dir.glob('*.jpg')))
    if existing >= IMAGES_PER_CLASS:
        print(f'  ✓ {brand:12} {model:10} — {existing} imgs (listo)')
        return existing

    downloaded = existing
    seen_hashes = set()

    for query in get_queries(brand, model):
        if downloaded >= IMAGES_PER_CLASS:
            break
        try:
            with DDGS() as ddgs:
                results = list(ddgs.images(
                    query,
                    max_results=30,
                    type_image='photo',
                    size='Medium',
                ))
        except Exception as e:
            print(f'    DDG error ({query[:30]}): {e}')
            time.sleep(3)
            continue

        for item in results:
            if downloaded >= IMAGES_PER_CLASS:
                break
            url = item.get('image', '')
            if not url:
                continue
            try:
                resp = requests.get(url, timeout=8, allow_redirects=True)
                if resp.status_code != 200:
                    continue
                content = resp.content
                # Deduplicate por hash MD5
                h = hashlib.md5(content).hexdigest()
                if h in seen_hashes:
                    continue
                seen_hashes.add(h)
                # Validar antes de guardar
                if not is_valid_car_image(content):
                    continue
                # Guardar como JPEG 224×224+
                img = Image.open(BytesIO(content)).convert('RGB')
                fname = class_dir / f'img_{downloaded:04d}.jpg'
                img.save(fname, 'JPEG', quality=92)
                downloaded += 1
            except Exception:
                continue

        time.sleep(1.5)   # Pausa entre queries para no ser bloqueado

    return downloaded

# ─── Descargar todas las clases ──────────────────────────────────
print('📥 Descargando imágenes de vehículos (DuckDuckGo)...\n')
summary = []
for brand, model in VEHICLE_CLASSES:
    print(f'⬇️  {brand} {model}...')
    n = download_class(brand, model)
    ok = n >= MIN_VALID
    summary.append((brand, model, n, ok))
    print(f'  {"✅" if ok else "⚠️ "} {brand} {model}: {n} imágenes válidas')

total = sum(n for _, _, n, _ in summary)
good  = sum(1 for _, _, n, ok in summary if ok)
print(f'\n{"="*55}')
print(f'Total de imágenes: {total}')
print(f'Clases con ≥{MIN_VALID} imgs: {good}/20')
if good < 15:
    print('⚠️  Menos de 15 clases tienen suficientes imágenes.')
    print('   El modelo puede tener accuracy bajo. Considera ejecutar de nuevo.')
else:
    print('✅ Dataset suficiente para entrenar.')

## 🧠 CELDA 5 — Entrenar EfficientNet-B0

⏱️ **Duración**: ~90 minutos con GPU T4  
☕ Puedes dejar esto corriendo y hacer otra cosa.

In [ ]:
!python ml/train.py \
    --data_dir ml/data/vehicleye_dataset \
    --epochs_phase1 5 \
    --epochs_phase2 10 \
    --output ml/checkpoints/efficientnet_b0_vehicleye.pth

from pathlib import Path
pth = Path('ml/checkpoints/efficientnet_b0_vehicleye.pth')
if pth.exists():
    print(f'\n✅ Modelo guardado: {pth}  ({pth.stat().st_size/1024/1024:.1f} MB)')
else:
    print('\n❌ No se generó el modelo — revisa errores arriba')

## 📊 CELDA 6 — Evaluar accuracy (opcional)

In [ ]:
!python ml/evaluate.py

from pathlib import Path
f = Path('reports/model_metrics.txt')
if f.exists():
    print('\n📊 MÉTRICAS:')
    print('='*60)
    print(f.read_text())

## 🔄 CELDA 7 — Exportar a ONNX

In [ ]:
!python ml/export_onnx.py \
    --checkpoint ml/checkpoints/efficientnet_b0_vehicleye.pth \
    --output     ml/checkpoints/vehicleye.onnx

from pathlib import Path
onnx = Path('ml/checkpoints/vehicleye.onnx')
if onnx.exists():
    print(f'\n✅ ONNX exportado: {onnx.stat().st_size/1024/1024:.1f} MB')
    print('   Render solo necesita este archivo (~50 MB, sin PyTorch).')
else:
    print('\n❌ No se generó vehicleye.onnx — revisa errores arriba')

## 💾 CELDA 8 — Descargar modelo

In [ ]:
from google.colab import files
from pathlib import Path

onnx = Path('ml/checkpoints/vehicleye.onnx')
if onnx.exists():
    print(f'📥 Descargando vehicleye.onnx ({onnx.stat().st_size/1024/1024:.1f} MB)...')
    files.download(str(onnx))
    print('\n✅ Revisa tu carpeta de Descargas — el archivo se descargó.')
else:
    print('❌ vehicleye.onnx no encontrado. Ejecuta CELDA 7 primero.')

## 🚀 CELDA 9 — Pasos siguientes para Render

In [ ]:
print("""
🎉 ¡ENTRENAMIENTO COMPLETADO!

Tienes: vehicleye.onnx (~50 MB) en tu carpeta de Descargas.

─────────────────────────────────────────────────────────
PASO A: Sube el modelo a GitHub Releases
─────────────────────────────────────────────────────────
 1. Abre: https://github.com/nick2331/Electiva_3/releases
 2. Click "Create a new release"
 3. Tag:   v1.0
 4. Title: Modelo VehiclEye v1.0
 5. Arrastra vehicleye.onnx al área de archivos
 6. Click "Publish release"

─────────────────────────────────────────────────────────
PASO B: Copia la URL del modelo
─────────────────────────────────────────────────────────
 En la release, clic derecho sobre vehicleye.onnx
 → "Copiar dirección del enlace"
 Ejemplo:
   https://github.com/nick2331/Electiva_3/releases/download/v1.0/vehicleye.onnx

─────────────────────────────────────────────────────────
PASO C: Configura en Render
─────────────────────────────────────────────────────────
 1. https://dashboard.render.com
 2. Servicio: vehicleye-api
 3. Pestaña: Environment
 4. Add variable:
      Key:   MODEL_DOWNLOAD_URL
      Value: (URL del paso B)
 5. Save → Render redeploya automáticamente

─────────────────────────────────────────────────────────
VERIFICACIÓN
─────────────────────────────────────────────────────────
 Panel admin → Estado del sistema → Modelo IA
 Debe decir: ✅ "Modelo ONNX cargado en memoria"

─────────────────────────────────────────────────────────
¡Listo! Las predicciones ahora son REALES. 🚗✅
""")